# Lab 8C — Audio RAG: Retrieval-Augmented Generation over an Audio Archive

### AIML ZG527 — Audio Analysis 

Authored by: Seetha Parameswaran, BITS Pilani WILP

Lab 8B ended on a wall of its own. CLAP's output is **one number** — a similarity score. A
scalar can **rank** clips, but it cannot **count** events, **order** them in time, handle
**negation**, or **explain** anything. To get an *answer in language*, we need a model that
**generates** language conditioned on audio evidence.

That is what **RAG** (Retrieval-Augmented Generation) provides. The insight is simple: a
bare LLM asked about an archive it has never seen will **hallucinate** a fluent, confident,
wrong answer. Rather than retrain the model on our data — expensive and constant — we
**retrieve** the relevant evidence and **hand it to the model to read**. This lab wires the
6B retriever to a small LLM, shows the difference grounding makes, and then probes exactly
where the approach still breaks.

**The RAG loop built in this lab:**

| Stage | What happens | Component |
|-------|--------------|-----------|
| Retrieve | Text question → CLAP → top-k relevant clips | CLAP (from 8B) |
| Augment | Turn retrieved clips into evidence, paste into the prompt | Evidence store |
| Generate | LLM answers **from the evidence only** | Flan-T5 |

**What you will see:**
- A bare LLM **hallucinating** answers about an archive it has no access to
- The same model, same question, **grounded** in retrieved evidence — side by side
- Why the retriever is the *honest* half: it **cannot invent**, only surface what exists
- Where RAG **fails** — counting, temporal order, and corpus-wide aggregation — and the root cause of each

---
### Learning Objectives
- Assemble a retrieve → augment → generate pipeline over an audio archive
- Demonstrate LLM hallucination, and how retrieval grounding suppresses it
- Build a textual evidence store from audio (the audio→text step)
- Diagnose RAG's failure modes and trace each to the evidence, not the retriever
- Explain why the audio→text step sets the quality ceiling of the whole system

---
### A note on the audio→text step

The slides describe two ways to turn retrieved audio into evidence: **caption the clips**
with a real audio captioning model, or **use existing text** attached to them (labels, tags,
metadata). This lab takes the second route, because no genuinely CPU-runnable audio
captioner exists — Qwen-Audio and SALMONN need a GPU. **The RAG architecture is identical
either way**; only the box labelled "audio→text" changes. Where production would call a
captioner, we read the metadata.

---
### Setup and Import Libraries
`!pip install librosa transformers pandas faiss-cpu --break-system-packages`

### Dataset: UrbanSound8K
Same corpus, **folds 1–2**. A smaller `MAX_CLIPS` is fine here — we do not need a large archive to make the point.

`> `UrbanSound8K.csv` and the `fold1/ … fold10/` directories sit directly under the dataset root.`

## 0. Setup

In [2]:
# If needed (uncomment):# 
# !pip install transformers faiss-cpu
import os, time, warnings
import numpy as np
import pandas as pd
import librosa
import torch
from transformers import ClapModel, ClapProcessor
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

warnings.filterwarnings("ignore")
np.random.seed(0)
torch.set_grad_enabled(False)

DATA_ROOT = "/home/seetha/PythonScriptsCourses/PythonScriptsCourses/Audio Analysis/Data/UrbanSound8k"
FOLDS     = [1, 2]
SR        = 48000
MAX_CLIPS = 400          # smaller than 6B -- we do not need a big corpus to make the point

2026-07-16 14:39:50.073977: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-07-16 14:39:50.074024: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [3]:
# ---- The retriever (from Lab 8B) ----
clap = ClapModel.from_pretrained("laion/clap-htsat-unfused").eval()
clap_proc = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")

# ---- The generator ----
LLM = "google/flan-t5-base"
tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForSeq2SeqLM.from_pretrained(LLM).eval()
print(f"Retriever: laion/clap-htsat-unfused")
print(f"Generator: {LLM} ({sum(p.numel() for p in llm.parameters())/1e6:.0f} M params)")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Retriever: laion/clap-htsat-unfused
Generator: google/flan-t5-base (248 M params)


In [4]:
def generate(prompt, max_new_tokens=60):    
    ids = tok(prompt, return_tensors="pt", truncation=True, max_length=512)    
    out = llm.generate(**ids, max_new_tokens=max_new_tokens)    
    return tok.decode(out[0], skip_special_tokens=True)

## 1. The problem: ask the bare LLM about audio it has never heard

The LLM has no access to our archive. It has never heard a single clip. Watch what it doeswhen we ask anyway.

In [5]:
QUESTIONS = [    
    "In the city sound archive, which recordings contain a dog barking?",    
    "How many jackhammer recordings are in fold 1 of the archive?",    
    "What is the most common sound in the archive?",]

for q in QUESTIONS:    
    print(f"Q: {q}")    
    print(f"A: {generate(q)}")    


Q: In the city sound archive, which recordings contain a dog barking?
A: snorting snort
Q: How many jackhammer recordings are in fold 1 of the archive?
A: 66
Q: What is the most common sound in the archive?
A: sonic boom


**Read those answers.** 

The model does not say *"I have no access to your archive."* It produces something **fluent, confident, and entirely invented**. This is hallucination,and it is the default behaviour — not an occasional glitch. The model's job is to produce plausible text, and it does that job perfectly. It was simply never given anything true towork from. 

**RAG's insight:** do not retrain the model. **Give it the evidence.**

## 2. Build the evidence store

Each clip needs a **textual** representation the LLM can read. Here, that is its metad
ata row. (In production: a generated caption.)

In [6]:
meta = pd.read_csv(os.path.join(DATA_ROOT, "UrbanSound8K.csv"))
meta = meta[meta["fold"].isin(FOLDS)].reset_index(drop=True)
if MAX_CLIPS:    
    meta = meta.sample(n=min(MAX_CLIPS, len(meta)), random_state=0).reset_index(drop=True)
    meta["path"] = meta.apply(    
        lambda r: os.path.join(DATA_ROOT, f"fold{r['fold']}", r["slice_file_name"]), axis=1)
    meta["duration"] = (meta["end"] - meta["start"]).round(1)
    SALIENCE = {1: "foreground", 2: "background"}
    
def to_evidence(row):    
    """Turn one clip into a sentence the LLM can read.       
    In production this line would call an audio CAPTIONING model."""    
    sound = row["class"].replace("_", " ")    
    return (f"Recording {row['slice_file_name']}: a {sound} sound, "            
            f"{row['duration']}s long, heard in the "            
            f"{SALIENCE.get(row['salience'], 'unknown')}, from fold {row['fold']}.")

meta["evidence"] = meta.apply(to_evidence, axis=1)
print(f"Evidence store: {len(meta)} entries\n")
for e in meta["evidence"].head(3):    
    print(" -", e)

Evidence store: 400 entries

 - Recording 203929-7-3-1.wav: a jackhammer sound, 4.0s long, heard in the background, from fold 2.
 - Recording 168713-9-0-46.wav: a street music sound, 4.0s long, heard in the foreground, from fold 2.
 - Recording 147926-0-0-36.wav: a air conditioner sound, 4.0s long, heard in the foreground, from fold 2.


## 3. RETRIEVE — CLAP finds the relevant clips

The retriever is exactly the machinery of Labs 8A and 8B. Nothing new. We embed the corpus with the **audio tower**, and the user's question with the **text tower**.

In [8]:
def l2n(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)


def embed_audio(paths, bs=8):
    vecs = []
    for i in range(0, len(paths), bs):
        wavs = [librosa.load(p, sr=SR, mono=True)[0] for p in paths[i:i+bs]]
        inp = clap_proc(audios=wavs, sampling_rate=SR, return_tensors="pt", padding=True)
        vecs.append(clap.get_audio_features(**inp).numpy())
    return l2n(np.vstack(vecs).astype("float32")) 

def embed_text(texts): 
    inp = clap_proc(text=texts, return_tensors="pt", padding=True)
    return l2n(clap.get_text_features(**inp).numpy().astype("float32"))


t0 = time.time()                                         
E = embed_audio(list(meta["path"]))             
print(f"Archive embedded: {E.shape} in {time.time()-t0:.1f}s")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Archive embedded: (400, 512) in 151.3s


In [9]:
def retrieve(question, k=5):    
    """Question (text) -> CLAP -> top-k clips -> their evidence sentences."""    
    q = embed_text([question])[0]    
    scores = E @ q    
    ids = np.argsort(-scores)[:k]    
    return ids, scores[ids]

ids, sc = retrieve("Are there any recordings of a dog barking?", k=5)
for i, s in zip(ids, sc):    
    print(f"  [{s:.3f}] {meta.loc[i, 'evidence']}")

  [0.708] Recording 31323-3-0-22.wav: a dog bark sound, 4.0s long, heard in the background, from fold 1.
  [0.676] Recording 164312-3-4-0.wav: a dog bark sound, 1.7s long, heard in the foreground, from fold 1.
  [0.668] Recording 101415-3-0-8.wav: a dog bark sound, 4.0s long, heard in the foreground, from fold 1.
  [0.668] Recording 197073-3-3-0.wav: a dog bark sound, 1.0s long, heard in the foreground, from fold 1.
  [0.668] Recording 203128-3-6-0.wav: a dog bark sound, 1.8s long, heard in the foreground, from fold 2.


Note the retriever **cannot invent anything**. It can only return clips that exist. That is precisely its value: it is the honest half of the system.

## 4 & 5. AUGMENT and GENERATE

Paste the retrieved evidence into the prompt, and ask the LLM to answer **from it**.

In [10]:
def rag_answer(question, k=5, verbose=True):
    ids, scores = retrieve(question, k=k)
    evidence = "\n".join(f"- {meta.loc[i, 'evidence']}" for i in ids)

    prompt = (
        "Answer the question using ONLY the evidence below. "
        "If the evidence does not contain the answer, say you do not know.\n\n"
        f"Evidence:\n{evidence}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    answer = generate(prompt)

    if verbose:
        print(f"Q: {question}\n")
        print("RETRIEVED EVIDENCE:")
        for i, s in zip(ids, scores):
            print(f"  [{s:.3f}] {meta.loc[i, 'evidence']}")
        print(f"\nA: {answer}\n")         
    return answer


_ = rag_answer("Are there any recordings of a dog barking?")   

Q: Are there any recordings of a dog barking?

RETRIEVED EVIDENCE:
  [0.708] Recording 31323-3-0-22.wav: a dog bark sound, 4.0s long, heard in the background, from fold 1.
  [0.676] Recording 164312-3-4-0.wav: a dog bark sound, 1.7s long, heard in the foreground, from fold 1.
  [0.668] Recording 101415-3-0-8.wav: a dog bark sound, 4.0s long, heard in the foreground, from fold 1.
  [0.668] Recording 197073-3-3-0.wav: a dog bark sound, 1.0s long, heard in the foreground, from fold 1.
  [0.668] Recording 203128-3-6-0.wav: a dog bark sound, 1.8s long, heard in the foreground, from fold 2.

A: Yes.



## 6. The comparison that mattersSame question. Same model. **The only difference is whether it was given evidence.**

In [11]:
TEST = [    
    "Is there a dog barking in the archive?",    
    "Are there any drilling sounds, and how long are they?",    
    "Is there a recording of a violin in the archive?",   # NOT in UrbanSound8K
]

for q in TEST:      
    print(f"Q: {q}\n")    
    print(f"  WITHOUT retrieval : {generate(q)}")    
    print(f"  WITH retrieval    : {rag_answer(q, verbose=False)}")   
    

Q: Is there a dog barking in the archive?

  WITHOUT retrieval : yes
  WITH retrieval    : yes
Q: Are there any drilling sounds, and how long are they?

  WITHOUT retrieval : yes
  WITH retrieval    : 4.0s long, heard in the foreground, from fold 1.
Q: Is there a recording of a violin in the archive?

  WITHOUT retrieval : yes
  WITH retrieval    : yes


**Look especially at the violin question.** There is no violin in UrbanSound8K.
- The bare LLM will happily invent one.
- The RAG system retrieves the nearest clips — none of which are violins — and, if the  prompt is doing its job, declines to confirm.
 
 **This is the whole point.** The LLM is no longer asked to *recall*. It is asked to *read*. Its answers become checkable against the evidence we can see in the printout above.

## 7. Where this still breaks

 Try these.

In [12]:
HARD = [    
    "How many times does the dog bark in the first recording?",    
    "Does the siren start before or after the children start playing?",    
    "How many dog barking recordings are in the archive in total?",]

for q in HARD:    
    _ = rag_answer(q, k=5)

Q: How many times does the dog bark in the first recording?

RETRIEVED EVIDENCE:
  [0.713] Recording 101415-3-0-8.wav: a dog bark sound, 4.0s long, heard in the foreground, from fold 1.
  [0.704] Recording 22347-3-0-0.wav: a dog bark sound, 1.1s long, heard in the foreground, from fold 2.
  [0.698] Recording 101415-3-0-2.wav: a dog bark sound, 4.0s long, heard in the foreground, from fold 1.
  [0.696] Recording 150341-3-1-0.wav: a dog bark sound, 0.4s long, heard in the foreground, from fold 1.
  [0.683] Recording 31323-3-0-22.wav: a dog bark sound, 4.0s long, heard in the background, from fold 1.

A: 4.0s long, heard in the foreground, from fold 1.

Q: Does the siren start before or after the children start playing?

RETRIEVED EVIDENCE:
  [0.592] Recording 123688-8-0-5.wav: a siren sound, 4.0s long, heard in the background, from fold 2.
  [0.566] Recording 91396-8-0-3.wav: a siren sound, 4.0s long, heard in the background, from fold 2.
  [0.532] Recording 108362-2-0-23.wav: a children

**Diagnose each failure — this is the most valuable part of the session.**

1. **"How many times does the dog bark?"** 
— The evidence store says *"a dog bark sound"*.   It contains **no count**. The retriever did its job perfectly; the *evidence itself* is   too impoverished. **RAG cannot recover information that was never extracted.**

2. **"Before or after?"** 
— Same root cause. Our evidence has no temporal structure. And   note: CLAP could not have supplied it either, because a mean-pooled vector has already   destroyed the ordering (Lab 6A, §2).

3. **"How many in total?"** 
— This is a **counting-over-the-whole-corpus** question. RAG   retrieves top-k. It structurally cannot answer aggregate questions — the LLM only ever   sees 5 rows.

**The lesson:** RAG grounds the generator in evidence. It does **not** create evidence. The quality ceiling of the whole system is set by the *audio -> text* step — which is why a real audio captioning model, or a true audio LM that ingests audio tokens directly, matters so much.

## Conclusion

In this lab we built a complete **audio RAG** system by wiring together the three pieces
developed across the session:

| Component | Role | Built in |
|-----------|------|----------|
| CLAP retriever | Search audio with a text question — **cannot invent** | 8A, 8B |
| Evidence store | Turn retrieved clips into text the LLM can read | 8C |
| LLM (Flan-T5) | Reason over the evidence and answer — **invents freely** | 8C |

**The one idea to take away:** a bare LLM asked about data it has never seen will
**hallucinate** — fluently and confidently. RAG does not retrain the model; it *grounds*
the model in retrieved evidence, so its answers become **checkable** against what we can
see was retrieved.

**And the limit we hit:** RAG grounds the generator, but it **cannot create evidence**.
When we asked *"how many times does the dog bark?"* or *"before or after?"*, the system
failed — not because retrieval failed, but because the *evidence itself* carried no counts
and no temporal order. **The quality ceiling of the whole system is set by the audio→text
step.** A richer captioning model, or a true audio LM that ingests audio tokens directly,
is what raises that ceiling.

This is the bridge to **Agentic AI for Audio** , where the LLM stops being a
passive answerer and starts **deciding** — when to retrieve, which tool to call, and what
to do next.